# EoMT COCO → Cityscapes — fine-tuning a due stadi (no cache)

Seguiamo le indicazioni del **project guide, Task 5**:

> *use AMP automatic mixed precision to reduce training time. A good first experiment to start out with is to fine-tune just the prediction head. Then you can gradually unfreeze the last layers and compare.*

**Perché niente cache dei backbone-token.** La vecchia pipeline salvava i token frozen di **una** vista augmentata e poi rileggeva le maschere con una **nuova** augmentation random (flip/scale/crop diversi): token e maschere disallineati → la mask/dice loss ottimizzava target sbagliati → mIoU che crollava (~14) dopo il picco iniziale. Qui usiamo il **forward normale** con augmentation live ad ogni step, in due stadi:
- **Stage 1**: encoder congelato, si allena solo la testa (query + mask_head + class_head).
- **Stage 2**: si scongelano gli ultimi N blocchi con LR basso + LLRD, partendo dai pesi dello Stage 1.

**Come si usa**: imposta tutto nella cella *Configurazione* (la prima di codice), poi esegui le celle in ordine. Ogni cella stampa un **log**; prima di ogni training c'è una **sanity-check** su dati e parametri. Gira sia su Colab sia su VS Code collegato a un kernel Colab (stessa VM, stesso filesystem).

## 1. Configurazione

**Unico punto da modificare.** Percorsi, ambiente e iperparametri dei due stadi. Le celle successive non vanno toccate per un uso normale.

In [7]:
"""Configurazione centrale del notebook.

Tutti i parametri modificabili stanno qui: repository, percorsi su Drive,
dataset, pesi/config e iperparametri dei due stadi di fine-tuning.
Le celle successive leggono solo da queste variabili.
"""
import os
import sys
import datetime
from pathlib import Path


def log(msg: str) -> None:
    """Stampa un messaggio con timestamp e flush immediato.

    Usato ovunque al posto di print() per un output ordinato e visibile in
    tempo reale sia su Colab sia su VS Code collegato a un kernel Colab.
    """
    print(f"[{datetime.datetime.now():%H:%M:%S}] {msg}", flush=True)


# --- Repository GitHub (clonata/aggiornata nella cella di setup) ---
GIT_USERNAME = "MottaDavide" # ChiaraApolito
REPO_NAME = "MaskArchitectureAnomaly_CourseProject"
BRANCH_NAME = "finetuning/coco-to-cityscapes"
PROJECT_FOLDER = "MaskArchitectureAnomaly_CourseProject"

# --- Percorsi su Google Drive ---
DRIVE_ROOT = Path("/content/drive/MyDrive")
LARGE_FILES = DRIVE_ROOT / "FAIML_project_and_presentation" / "01_Project" / "large_files"
PROJECT_PATH = DRIVE_ROOT / PROJECT_FOLDER          # repo clonata su Drive
PROJECT_ROOT = PROJECT_PATH / "eomt"                # package Python del modello
REPO_URL = f"https://github.com/{GIT_USERNAME}/{REPO_NAME}.git"

# --- Dataset Cityscapes ---
DRIVE_DATA_DIR = LARGE_FILES / "datasets" / "cityscapes"   # zip originali su Drive
LOCAL_DATA_DIR = Path("/content/cityscapes")               # copia locale (disco VM, più veloce)
USE_LOCAL_DATA_COPY = True
DATA_DIR = LOCAL_DATA_DIR if USE_LOCAL_DATA_COPY else DRIVE_DATA_DIR

# --- Pesi pre-addestrati e config di fine-tuning ---
COCO_WEIGHTS = LARGE_FILES / "weights" / "eomt_coco.bin"   # checkpoint EoMT pre-addestrato su COCO
FT_CONFIG_DIR = PROJECT_ROOT / "configs" / "dinov2" / "finetuning"
STAGE1_CONFIG = FT_CONFIG_DIR / "coco_to_cityscapes_freeze_backbone.yaml"   # head-only
STAGE2_CONFIG = FT_CONFIG_DIR / "coco_to_cityscapes_unfreeze_last.yaml"     # unfreeze last N

# --- Cartelle di output dei checkpoint ---
CKPT_ROOT = LARGE_FILES / "weights" / "finetuned"
STAGE1_CKPT_DIR = CKPT_ROOT / "coco_to_cityscapes_stage1_head"
STAGE2_CKPT_DIR = CKPT_ROOT / "coco_to_cityscapes_stage2_unfreeze_last"

# --- Iperparametri di training ---
# La risoluzione si imposta SOLO via data.img_size: main.py la propaga a
# model/network/encoder con link_arguments (gli override su model.* sono ignorati).
IMG_SIZE = [512, 512]   # 512 invece di 1024 -> ~1/4 del calcolo, batch più grandi su L4

# Precisione AMP, compilazione e early stopping
PRECISION = "bf16-mixed"      # bf16 piu' stabile di fp16 su dice/focal (L4 supporta bf16)
USE_COMPILE = True            # torch.compile del modello prima del fit (training piu' veloce)
EARLY_STOP_PATIENCE = 5       # epoche senza miglioramento di val_iou_all prima di fermarsi

# Knobs statici di ogni stadio (i pesi iniziali dello Stage 2 sono runtime).
STAGE1 = {
    "config": STAGE1_CONFIG,
    "ckpt_dir": STAGE1_CKPT_DIR,
    "run_name": "coco_to_cityscapes_stage1_head",
    "epochs": 15,
    "batch_size": 8,           # @512px su L4; scendi a 4 se vai OOM
    "load_class_head": False,  # class_head random: Cityscapes ha 19 classi != COCO
}
STAGE2 = {
    "config": STAGE2_CONFIG,
    "ckpt_dir": STAGE2_CKPT_DIR,
    "run_name": "coco_to_cityscapes_stage2_unfreeze_last",
    "epochs": 20,
    "batch_size": 8,           # con grad sugli ultimi blocchi potresti dover scendere a 4
    "load_class_head": True,   # riparte dalla testa già allenata nello Stage 1
}

log("Configurazione caricata.")
log(f"  PROJECT_ROOT = {PROJECT_ROOT}")
log(f"  DATA_DIR     = {DATA_DIR}  (copia locale={USE_LOCAL_DATA_COPY})")
log(f"  IMG_SIZE     = {IMG_SIZE}")
log(f"  STAGE1: {STAGE1['epochs']} ep, batch {STAGE1['batch_size']}  |  "
    f"STAGE2: {STAGE2['epochs']} ep, batch {STAGE2['batch_size']}")

[09:10:52] Configurazione caricata.
[09:10:52]   PROJECT_ROOT = /content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject/eomt
[09:10:52]   DATA_DIR     = /content/cityscapes  (copia locale=True)
[09:10:52]   IMG_SIZE     = [512, 512]
[09:10:52]   STAGE1: 15 ep, batch 8  |  STAGE2: 20 ep, batch 8


## 2. Setup ambiente + repository

Rileva Colab, monta Drive (idempotente), clona/aggiorna la repo, crea le cartelle di output, verifica gli input e installa le dipendenze.

In [8]:
"""Setup ambiente: Colab/Drive, repository, cartelle, dipendenze."""
import shutil

# Colab e VS Code+kernel-Colab condividono la stessa VM: google.colab è disponibile in entrambi.
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
log(f"IN_COLAB = {IN_COLAB}")

if IN_COLAB:
    drive.mount("/content/drive")   # idempotente: se già montato non fa nulla
    log("Drive montato.")
else:
    log("Kernel non-Colab: salto drive.mount (verifica che i percorsi siano accessibili).")


def setup_repository() -> None:
    """Clona la repo se assente, altrimenti la riallinea al branch remoto."""
    if not PROJECT_PATH.exists():
        log("Clonazione repository...")
        os.system(f'git clone --branch "{BRANCH_NAME}" "{REPO_URL}" "{PROJECT_PATH}"')
    elif not (PROJECT_PATH / ".git").exists():
        log("Cartella esistente ma non e' una repo Git: la ricreo...")
        shutil.rmtree(PROJECT_PATH)
        os.system(f'git clone --branch "{BRANCH_NAME}" "{REPO_URL}" "{PROJECT_PATH}"')
    else:
        log("Aggiornamento repository al branch remoto...")
        os.chdir(PROJECT_PATH)
        os.system("git fetch origin")
        os.system(f"git checkout {BRANCH_NAME}")
        os.system(f"git reset --hard origin/{BRANCH_NAME}")
    log("Repository pronta.")


setup_repository()

# Rende importabile il package 'eomt' e ci si posiziona dentro (per i path relativi dei config).
assert PROJECT_ROOT.exists(), f"PROJECT_ROOT non trovato: {PROJECT_ROOT}"
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
log(f"PWD = {Path.cwd()}")

# Crea le cartelle di output e verifica che gli input esistano.
STAGE1_CKPT_DIR.mkdir(parents=True, exist_ok=True)
STAGE2_CKPT_DIR.mkdir(parents=True, exist_ok=True)
assert COCO_WEIGHTS.exists(), f"Pesi COCO mancanti: {COCO_WEIGHTS}"
assert STAGE1_CONFIG.exists() and STAGE2_CONFIG.exists(), "Config di finetuning mancanti."

log("Installazione dipendenze (eomt/requirements.txt)...")
get_ipython().system("pip install -q -r requirements.txt")
log("Setup completato.")

[09:10:52] IN_COLAB = True


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[09:10:56] Drive montato.
[09:10:56] Aggiornamento repository al branch remoto...
[09:10:58] Repository pronta.
[09:10:58] PWD = /content/drive/MyDrive/MaskArchitectureAnomaly_CourseProject/eomt
[09:10:58] Installazione dipendenze (eomt/requirements.txt)...
[09:11:03] Setup completato.


## 3. Dataset Cityscapes su disco locale

Copia gli zip su `/content` (disco della VM) per ridurre la latenza di I/O. Vale sia per Colab sia per VS Code+kernel-Colab.

In [9]:
"""Copia gli zip di Cityscapes sul disco locale della VM e verifica che ci siano."""
import shutil

if USE_LOCAL_DATA_COPY:
    LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
    for name in ["leftImg8bit_trainvaltest.zip", "gtFine_trainvaltest.zip"]:
        src, dst = DRIVE_DATA_DIR / name, LOCAL_DATA_DIR / name
        assert src.exists(), f"File mancante su Drive: {src}"
        if dst.exists() and dst.stat().st_size == src.stat().st_size:
            log(f"Gia' copiato: {dst.name} ({dst.stat().st_size/1e9:.2f} GB)")
        else:
            log(f"Copio {src.name} -> {dst} ...")
            shutil.copy2(src, dst)
            log(f"OK: {dst.name} ({dst.stat().st_size/1e9:.2f} GB)")
else:
    log("Uso il dataset direttamente da Drive.")

for name in ["leftImg8bit_trainvaltest.zip", "gtFine_trainvaltest.zip"]:
    assert (DATA_DIR / name).exists(), f"Zip mancante in DATA_DIR: {name}"
log("Dataset pronto.")

[09:11:04] Gia' copiato: leftImg8bit_trainvaltest.zip (11.59 GB)
[09:11:04] Gia' copiato: gtFine_trainvaltest.zip (0.25 GB)
[09:11:04] Dataset pronto.


## 4. Login WandB

In [10]:
"""Imposta la WandB API key (chiesta in modo sicuro se non gia' nell'ambiente)."""
from getpass import getpass

if not os.environ.get("WANDB_API_KEY"):
    os.environ["WANDB_API_KEY"] = getpass("WandB API key: ")
log("WANDB_API_KEY impostata." if os.environ.get("WANDB_API_KEY") else "WANDB_API_KEY NON impostata!")

[09:11:04] WANDB_API_KEY impostata.


## 5. Funzioni di supporto

`build_cli` costruisce trainer/modello/datamodule dal config; `sanity_check_data` valida i dati; `run_stage` allena; `export_weights` salva i pesi puliti; `fit_stage` mette insieme tutto per uno stadio.

In [ ]:
"""Funzioni di supporto: build del trainer, sanity-check, training, export pesi."""
import re
import torch
from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor, EarlyStopping

try:
    from main import LightningCLI   # subclass del progetto (link_arguments, override)
except ImportError:
    from lightning.pytorch.cli import LightningCLI

DEVICE_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
log(f"Device: {DEVICE_NAME}")
if torch.cuda.is_available():
    log(f"VRAM totale: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


def build_cli(config_path, ckpt_dir, run_name, max_epochs, batch_size,
              ckpt_path, load_ckpt_class_head, limit_val_batches=1.0):
    """Costruisce model/datamodule/trainer dal config via ``LightningCLI(run=False)``.

    Reinserisce AMP e gradient clipping che i ``trainer_defaults`` di ``main.py``
    applicano solo con ``cli_main`` (qui usiamo run=False). La risoluzione si passa
    via ``--data.img_size`` perche' e' l'unico override che si propaga a
    model/network/encoder.
    """
    args = [
        "-c", str(config_path),
        "--trainer.accelerator", "gpu",
        "--trainer.devices", "1",
        "--trainer.max_epochs", str(max_epochs),
        "--trainer.precision", PRECISION,                  # AMP (bf16-mixed)
        "--trainer.gradient_clip_val", "0.01",             # stabilita' (come upstream)
        "--trainer.gradient_clip_algorithm", "norm",
        "--trainer.log_every_n_steps", "10",
        "--trainer.check_val_every_n_epoch", "1",
        "--trainer.limit_val_batches", str(limit_val_batches),
        "--trainer.num_sanity_val_steps", "0",
        "--trainer.default_root_dir", str(ckpt_dir),
        "--trainer.logger.init_args.project", "eomt",
        "--trainer.logger.init_args.name", run_name,
        "--trainer.logger.init_args.resume", "allow",
        "--data.path", str(DATA_DIR),
        "--data.batch_size", str(batch_size),
        "--data.img_size", str(IMG_SIZE),
        "--model.init_args.ckpt_path", str(ckpt_path),
        "--model.init_args.load_ckpt_class_head", str(load_ckpt_class_head).lower(),
    ]
    orig = sys.argv
    sys.argv = [sys.argv[0]]
    try:
        cli = LightningCLI(args=args, run=False, save_config_callback=None)
    finally:
        sys.argv = orig
    log(f"CLI costruita | run='{run_name}' | epochs={max_epochs} | batch={batch_size}")
    return cli


def _group(name):
    """Collassa il nome di un parametro al suo modulo (un rigo per blocco transformer)."""
    m = re.search(r"(.*blocks\.\d+)", name)
    return m.group(1) if m else name.rsplit(".", 1)[0]


def print_model_summary(model):
    """Logga img_size, flag della rete e i parametri trainable raggruppati per modulo."""
    net = model.network
    log(f"img_size={model.img_size} | num_classes={model.num_classes} | "
        f"masked_attn={net.masked_attn_enabled} | num_q={net.num_q} | num_blocks={net.num_blocks}")
    trainable, total, groups = 0, 0, {}
    for n, p in model.named_parameters():
        total += p.numel()
        if p.requires_grad:
            trainable += p.numel()
            groups[_group(n)] = groups.get(_group(n), 0) + p.numel()
    log(f"Parametri trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
    for k in sorted(groups):
        log(f"  [train] {k}  ({groups[k]:,})")


def _img_size_tuple(model):
    """img_size del modello come tupla (gestisce int o lista)."""
    s = model.img_size
    return (s, s) if isinstance(s, int) else tuple(s)


def sanity_check_data(cli):
    """Verifica la pipeline dati prima del training.

    Stampa le dimensioni di train/val e le statistiche di un batch reale, poi
    controlla che la risoluzione dei dati combaci col modello e che ogni maschera
    abbia la sua label (immagine e maschere dalla STESSA augmentation: e' proprio
    l'allineamento che la vecchia cache rompeva).
    """
    dm = cli.datamodule
    dm.setup("fit")
    train_loader, val_loader = dm.train_dataloader(), dm.val_dataloader()
    log(f"train: {len(train_loader.dataset)} img / {len(train_loader)} batch | "
        f"val: {len(val_loader.dataset)} img | batch_size={val_loader.batch_size}")
    imgs, targets = next(iter(train_loader))
    log(f"batch imgs: shape={tuple(imgs.shape)} dtype={imgs.dtype} "
        f"range=[{imgs.min():.0f},{imgs.max():.0f}]")
    t0 = targets[0]
    log(f"target[0]: keys={list(t0.keys())} | n_oggetti={t0['masks'].shape[0]} | "
        f"masks={tuple(t0['masks'].shape)} {t0['masks'].dtype} | labels={t0['labels'].tolist()}")
    assert tuple(imgs.shape[-2:]) == _img_size_tuple(cli.model), "img_size dati != modello!"
    assert t0["masks"].shape[0] == t0["labels"].shape[0] > 0, "maschere/label vuote o disallineate!"
    log("OK sanity dati: immagine e maschere provengono dalla stessa augmentation.")


def run_stage(cli, ckpt_dir):
    """Aggiunge ModelCheckpoint + LearningRateMonitor e lancia il fit. Ritorna il best ckpt."""
    trainer = cli.trainer
    trainer.callbacks = [cb for cb in trainer.callbacks
                         if not isinstance(cb, (ModelCheckpoint, LearningRateMonitor, EarlyStopping))]
    ckpt_cb = ModelCheckpoint(
        dirpath=str(ckpt_dir), filename="best",
        monitor="metrics/val_iou_all", mode="max",
        save_top_k=1, save_last=True, auto_insert_metric_name=False,
    )
    early_cb = EarlyStopping(monitor="metrics/val_iou_all", mode="max",
                             patience=EARLY_STOP_PATIENCE)
    trainer.callbacks += [ckpt_cb, LearningRateMonitor(logging_interval="epoch"), early_cb]
    log("Inizio training...")
    model = torch.compile(cli.model) if USE_COMPILE else cli.model   # compile -> training piu' veloce
    trainer.fit(model, datamodule=cli.datamodule)   # dataloader standard, augmentation live
    best_iou = float(ckpt_cb.best_model_score) * 100 if ckpt_cb.best_model_score is not None else float("nan")
    log(f"Training finito. best mIoU={best_iou:.2f} | ckpt={ckpt_cb.best_model_path}")
    return ckpt_cb.best_model_path


def export_weights(ckpt_path, out_bin):
    """Estrae lo state_dict da un .ckpt Lightning in un .bin pulito (caricabile con weights_only=True)."""
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state_dict = ckpt.get("state_dict", ckpt)
    state_dict = {k: v for k, v in state_dict.items() if "criterion.empty_weight" not in k}
    torch.save(state_dict, out_bin)
    log(f"Pesi esportati in: {out_bin}")
    return out_bin


def fit_stage(stage, init_weights):
    """Pipeline completa di uno stadio: build -> summary -> sanity -> fit.

    ``stage`` e' uno dei dict STAGE1/STAGE2 della cella di configurazione;
    ``init_weights`` sono i pesi di partenza (COCO per lo Stage 1, output dello
    Stage 1 per lo Stage 2). Ritorna ``(cli, best_ckpt_path)``.
    """
    cli = build_cli(
        config_path=stage["config"], ckpt_dir=stage["ckpt_dir"],
        run_name=stage["run_name"], max_epochs=stage["epochs"],
        batch_size=stage["batch_size"], ckpt_path=init_weights,
        load_ckpt_class_head=stage["load_class_head"],
    )
    print_model_summary(cli.model)
    sanity_check_data(cli)
    best = run_stage(cli, stage["ckpt_dir"])
    return cli, best

## 6. Stage 1 — fine-tune della sola testa (encoder congelato)

La `class_head` parte random (`load_class_head=False`) perché Cityscapes ha 19 classi != COCO. Con l'encoder congelato adattiamo la testa senza rovinare il decoder pre-addestrato.

In [12]:
# Stage 1: encoder congelato, si allena solo la testa. Parte dai pesi COCO.
cli1, best1_ckpt = fit_stage(STAGE1, init_weights=COCO_WEIGHTS)

# Pesi puliti da cui ripartira' lo Stage 2.
stage1_bin = export_weights(best1_ckpt, STAGE1_CKPT_DIR / "stage1_weights.bin")

INFO: Seed set to 0
INFO:lightning.fabric.utilities.seed:Seed set to 0
INFO:root:Interpolated pos_embed from 1600 to 1024 tokens
INFO:root:Loaded 195 keys
INFO: Using bfloat16 Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using bfloat16 Automatic Mixed Precision (AMP)
INFO: Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
INFO:lightning.pytorch.utilities.rank_zero:Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank

[09:11:07] CLI costruita | run='coco_to_cityscapes_stage1_head' | epochs=15 | batch=8
[09:11:07] img_size=(512, 512) | num_classes=19 | masked_attn=True | num_q=200 | num_blocks=3
[09:11:07] Parametri trainable: 6,677,780 / 93,133,076 (7.17%)
[09:11:07]   [train] network.class_head  (15,380)
[09:11:07]   [train] network.mask_head.0  (590,592)
[09:11:07]   [train] network.mask_head.2  (590,592)
[09:11:07]   [train] network.mask_head.4  (590,592)
[09:11:07]   [train] network.q  (153,600)
[09:11:07]   [train] network.upscale.0.conv1  (2,360,064)
[09:11:07]   [train] network.upscale.0.conv2  (6,912)
[09:11:07]   [train] network.upscale.0.norm  (1,536)
[09:11:07]   [train] network.upscale.1.conv1  (2,360,064)
[09:11:07]   [train] network.upscale.1.conv2  (6,912)
[09:11:07]   [train] network.upscale.1.norm  (1,536)


AttributeError: 'CityscapesSemantic' object has no attribute 'batch_size'

## 7. Stage 2 — scongela gli ultimi blocchi

Parte dai pesi dello Stage 1 (`load_class_head=True`: la testa è già a 19 classi) con optimizer/scheduler **puliti** (no resume → `total_steps` del poly schedule corretto). LR basso + LLRD per non distruggere le feature DINOv2.

In [ ]:
# Stage 2: scongela gli ultimi blocchi. Parte dai pesi (puliti) dello Stage 1.
cli2, best2_ckpt = fit_stage(STAGE2, init_weights=stage1_bin)
log(f"Checkpoint finale fine-tuned: {best2_ckpt}")

## 8. Validation finale

La mIoU è loggata su WandB (`metrics/val_iou_all`) e stampata a fine epoca. Per il confronto richiesto dalla guida (modello fine-tunato vs. EoMT-Cityscapes fornito), valuta entrambi sull'intero val set con la **stessa pipeline** del Task 4.

In [ ]:
# Rivalida il best dello Stage 2 (mIoU sull'intero val set).
log("Rivalidazione del best checkpoint dello Stage 2...")
cli2.trainer.validate(cli2.model, datamodule=cli2.datamodule, ckpt_path=best2_ckpt)